# baseline v4 추천 수정본 (오프라인)

기존 v3를 기반으로 다음을 수정했습니다.

- assistant 정답 부분에만 loss 적용 (prompt/image/padding loss masking)
- Qwen2.5-VL 동적 해상도 사용, OCR/작은 글자 보존 강화
- BF16/FP16 dtype 일관화
- 동일 question group이 validation에 누출되지 않도록 split
- 마지막 gradient accumulation 잔여 batch도 optimizer step 수행
- `generate()+문자열 파싱` 대신 a/b/c/d next-token logit scoring
- validation loss뿐 아니라 실제 대회 지표인 Accuracy 측정
- sample_submission 형식을 그대로 사용

`TRAIN_LIMIT=None`이 실제 제출용 설정이며, 먼저 200 정도로 smoke test 후 전체 학습을 권장합니다.



# 환경 준비

미리 다운로드해 둔 `downloads/libs` 폴더의 wheel 파일로 라이브러리를 설치합니다. **인터넷에 접속하지 않습니다.**

- 아래 셀 실행
- ipykernel 설치
- 아래 셀 다시 실행 : 무한 로딩 시 restart
- hello 출력시 경로 설정 셀 → torch 설치


In [1]:
print('hello123')

hello123


In [2]:
import os

# 사전 다운로드 폴더 (다운로드 노트북과 동일한 구조)
ASSET_DIR  = "downloads"
LIB_DIR    = os.path.join(ASSET_DIR, "libs")                              # 라이브러리 wheel
MODEL_DIR  = os.path.join(ASSET_DIR, "models", "Qwen2.5-VL-3B-Instruct")  # 사전학습 모델
DATA_DIR   = "dataset"                                                     # 현재 작업 폴더의 데이터셋
OUTPUT_DIR = "output"                                                      # 학습 결과, 제출 파일
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Hugging Face 오프라인 모드 : 모델 로드 시 인터넷 접속 시도 자체를 차단
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

# 사전 준비 폴더 확인
for name, path in [("라이브러리", LIB_DIR), ("모델", MODEL_DIR), ("데이터", DATA_DIR)]:
    state = "OK" if os.path.isdir(path) and os.listdir(path) else "없음 → 사전 준비 필요"
    print(f"{name:<5s} {state:<14s} {os.path.abspath(path)}")


라이브러리 OK             c:\SSAFY\AIChallenge\downloads\libs
모델    OK             c:\SSAFY\AIChallenge\downloads\models\Qwen2.5-VL-3B-Instruct
데이터   OK             c:\SSAFY\AIChallenge\data


In [3]:
# 미리 다운로드한 wheel 파일로 설치 (인터넷 미사용). CUDA 12.8 빌드로 버전 고정
%pip install --no-index --find-links downloads/libs torch==2.11.0+cu128 torchvision==0.26.0+cu128 torchaudio==2.11.0+cu128

Looking in links: downloads/libs
Note: you may need to restart the kernel to use updated packages.


In [4]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())
assert "+cu" in torch.__version__, (
    f"CPU 전용 torch({torch.__version__})가 설치되어 있습니다. "
    "downloads/libs 폴더에 torch-*+cu128 wheel 이 있는지 확인하고 위 설치 셀을 다시 실행하세요."
)
print(torch.cuda.get_device_name())

2.11.0+cu128
True
NVIDIA GeForce RTX 5060 Ti


In [5]:
# 미리 다운로드한 wheel 파일로 설치 (인터넷 미사용)
%pip -q install --no-index --find-links downloads/libs "transformers>=4.43.2,<5.0.0" "accelerate>=0.34.2" "peft>=0.13.2" "bitsandbytes>=0.43.3" datasets pillow pandas --upgrade

Note: you may need to restart the kernel to use updated packages.


# 데이터 준비

데이터셋은 사전에 배포되어 `data` 폴더에 아래 구조로 준비되어 있어야 합니다. 다운로드·압축 해제 작업은 없습니다.

- data/train.csv, data/train 폴더
- data/test.csv, data/test 폴더
- data/sample_submission.csv


# 라이브러리, 데이터, 설정

In [6]:
import os, re, math, random
import numpy as np
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
import torch
from typing import Any
from transformers import (
    AutoModelForVision2Seq,
    AutoProcessor,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from tqdm import tqdm

Image.MAX_IMAGE_PIXELS = None

device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "이 노트북은 CUDA GPU 실행을 전제로 합니다."
print("Device:", device)

MODEL_ID = MODEL_DIR
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Qwen2.5-VL의 동적 해상도를 사용합니다.
# 텍스트/OCR 문제를 고려해 기존 384x384 상당보다 더 많은 픽셀을 보존합니다.
# OOM이면 MAX_PIXELS를 512*28*28 수준까지 낮추세요.
MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 768 * 28 * 28

# 빠른 동작 확인 때만 200 등으로 설정. 실제 제출용 학습은 None 권장.
TRAIN_LIMIT = None
VAL_FRAC = 0.10

train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_df  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

if TRAIN_LIMIT is not None:
    train_df = train_df.sample(n=min(TRAIN_LIMIT, len(train_df)), random_state=SEED).reset_index(drop=True)

print("train:", len(train_df), "test:", len(test_df))
print(train_df["answer"].value_counts(normalize=True).sort_index())


c:\SSAFY\AIChallenge\baseline\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0921 09:52:02.047000 10260 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Device: cuda
train: 6714 test: 6714
answer
a    0.253202
b    0.244861
c    0.255585
d    0.246351
Name: proportion, dtype: float64


# 모델, Processor

사전에 다운로드한 로컬 모델(`downloads/models/Qwen2.5-VL-3B-Instruct`)을 로드합니다. 인터넷 다운로드는 진행되지 않으며, 디스크에서 로드하는 데 1~3분 정도가 소요됩니다.

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - LoRA 구현 : LoraConfig()

In [7]:
# BF16을 지원하면 4bit 계산 dtype과 autocast dtype을 일치시킵니다.
COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print("compute dtype:", COMPUTE_DTYPE)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
    trust_remote_code=True,
    local_files_only=True,
)
processor.tokenizer.padding_side = "right"

base_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=COMPUTE_DTYPE,
    trust_remote_code=True,
    local_files_only=True,
)

# gradient checkpointing 학습 시 cache는 끕니다.
base_model.config.use_cache = False
base_model = prepare_model_for_kbit_training(base_model)
base_model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

# device_map="auto" + 4bit 모델에는 model.to(device)를 다시 호출하지 않습니다.
MODEL_DEVICE = model.device
print("model device:", MODEL_DEVICE)


The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


compute dtype: torch.bfloat16


c:\SSAFY\AIChallenge\baseline\Lib\site-packages\transformers\models\auto\modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:20<00:00, 10.22s/it]


trainable params: 18,576,384 || all params: 3,773,199,360 || trainable%: 0.4923
model device: cuda:0


# 프롬프트 템플릿

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - 프롬프트 템플릿 : convert_to_chatml(), formatting_prompts_func()

In [8]:
SYSTEM_INSTRUCT = (
    "이미지를 보고 객관식 질문에 답하세요. "
    "필요하면 이미지 속 글자, 숫자, 표지판, 가격, 상호명 등 세부 정보를 주의 깊게 읽으세요. "
    "최종 답은 반드시 a, b, c, d 중 하나의 소문자 한 글자만 출력하세요."
)

def build_mc_prompt(question, a, b, c, d):
    return (
        f"질문: {question}\n"
        f"(a) {a}\n"
        f"(b) {b}\n"
        f"(c) {c}\n"
        f"(d) {d}\n"
        "정답:"
    )


# Custom Dataset, Collator

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - TensorDataset()

    챕터 5-2 데이터 생성 및 파인튜닝 (향후 학습 분량)
    - IntentDataset()

In [9]:
class VQAMCDataset(Dataset):
    def __init__(self, df, processor, train=True):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.train = train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        image_path = os.path.join(DATA_DIR, row["path"])
        with Image.open(image_path) as im:
            img = im.convert("RGB")

        user_text = build_mc_prompt(
            str(row["question"]), str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])
        )

        messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
            {"role": "user", "content": [
                {"type": "image", "image": img},
                {"type": "text", "text": user_text},
            ]},
        ]
        if self.train:
            gold = str(row["answer"]).strip().lower()
            messages.append({"role": "assistant", "content": [{"type": "text", "text": gold}]})

        return {"messages": messages, "image": img}


def _find_last_subsequence(seq, pattern):
    """seq에서 pattern의 마지막 시작 위치를 찾습니다."""
    n = len(pattern)
    for i in range(len(seq) - n, -1, -1):
        if seq[i:i+n] == pattern:
            return i
    return -1


@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __post_init__(self):
        # Qwen ChatML의 assistant role 시작 지점. 이 뒤만 loss를 계산합니다.
        self.assistant_prefix_ids = self.processor.tokenizer.encode(
            "<|im_start|>assistant\n", add_special_tokens=False
        )

    def __call__(self, batch):
        texts, images = [], []
        for sample in batch:
            text = self.processor.apply_chat_template(
                sample["messages"],
                tokenize=False,
                add_generation_prompt=False,
            )
            texts.append(text)
            images.append(sample["image"])

        enc = self.processor(
            text=texts,
            images=images,
            padding=True,
            return_tensors="pt",
        )

        if self.train:
            # 핵심 수정: system/user/image/padding에는 loss를 걸지 않고
            # 마지막 assistant 답변(a/b/c/d)에만 supervision을 줍니다.
            labels = torch.full_like(enc["input_ids"], -100)

            for i in range(len(batch)):
                valid_len = int(enc["attention_mask"][i].sum().item())
                seq = enc["input_ids"][i, :valid_len].tolist()
                pos = _find_last_subsequence(seq, self.assistant_prefix_ids)
                if pos < 0:
                    raise RuntimeError("assistant prefix를 찾지 못했습니다. chat template/tokenizer를 확인하세요.")
                answer_start = pos + len(self.assistant_prefix_ids)
                labels[i, answer_start:valid_len] = enc["input_ids"][i, answer_start:valid_len]

            enc["labels"] = labels

        return enc


# DataLoader

#### 실습 참고 내용

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 데이터로더 정의 : DataLoader()

In [10]:
def normalize_question(x):
    return re.sub(r"\s+", " ", str(x).strip().lower())


def make_group_split(df, val_frac=0.10, seed=42, trials=100):
    """
    동일/유사(공백 정규화) question이 train/valid 양쪽에 동시에 들어가지 않도록 group split.
    추가 라이브러리 없이 여러 셔플 중 정답 분포가 가장 잘 맞는 split을 선택합니다.
    """
    groups = {}
    for idx, q in enumerate(df["question"].map(normalize_question)):
        groups.setdefault(q, []).append(idx)

    group_items = list(groups.items())
    target_n = int(round(len(df) * val_frac))
    overall = df["answer"].value_counts(normalize=True).reindex(list("abcd"), fill_value=0.0)

    best = None
    for t in range(trials):
        rng = random.Random(seed + t)
        items = group_items.copy()
        rng.shuffle(items)

        val_idx = []
        for _, idxs in items:
            if len(val_idx) >= target_n:
                break
            val_idx.extend(idxs)

        val_idx = sorted(set(val_idx))
        val_prop = df.iloc[val_idx]["answer"].value_counts(normalize=True).reindex(list("abcd"), fill_value=0.0)
        score = float((val_prop - overall).abs().sum()) + abs(len(val_idx) - target_n) / len(df)

        if best is None or score < best[0]:
            best = (score, val_idx)

    val_idx = set(best[1])
    train_idx = [i for i in range(len(df)) if i not in val_idx]
    val_idx = sorted(val_idx)

    return df.iloc[train_idx].reset_index(drop=True), df.iloc[val_idx].reset_index(drop=True)


train_subset, valid_subset = make_group_split(train_df, VAL_FRAC, SEED)
print("train/valid:", len(train_subset), len(valid_subset))
print("valid answer dist:\n", valid_subset["answer"].value_counts(normalize=True).sort_index())

train_ds = VQAMCDataset(train_subset, processor, train=True)
valid_ds = VQAMCDataset(valid_subset, processor, train=True)

train_loader = DataLoader(
    train_ds,
    batch_size=1,
    shuffle=True,
    collate_fn=DataCollator(processor, True),
    num_workers=0,
    pin_memory=True,
)
valid_loader = DataLoader(
    valid_ds,
    batch_size=1,
    shuffle=False,
    collate_fn=DataCollator(processor, True),
    num_workers=0,
    pin_memory=True,
)


train/valid: 6043 671
valid answer dist:
 answer
a    0.250373
b    0.242921
c    0.260805
d    0.245902
Name: proportion, dtype: float64


# fine-tuning

- 200개만 학습 : 10~20분 소요

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - 모델 정의 : SimpleMLP(), SequentialMLP()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [11]:
from tqdm.auto import tqdm

GRAD_ACCUM = 8
EPOCHS = 1
LR = 5e-5
WEIGHT_DECAY = 0.01

trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=LR, weight_decay=WEIGHT_DECAY)

updates_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
num_training_steps = EPOCHS * updates_per_epoch
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=max(1, int(num_training_steps * 0.05)),
    num_training_steps=num_training_steps,
)

use_scaler = (COMPUTE_DTYPE == torch.float16)
scaler = torch.amp.GradScaler("cuda", enabled=use_scaler)

optimizer.zero_grad(set_to_none=True)
global_step = 0

for epoch in range(EPOCHS):
    model.train()
    running_raw_loss = 0.0
    accum_count = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1} [train]", unit="batch")

    for step, batch in enumerate(progress_bar, start=1):
        batch = {k: v.to(MODEL_DEVICE, non_blocking=True) for k, v in batch.items()}

        with torch.autocast(device_type="cuda", dtype=COMPUTE_DTYPE):
            outputs = model(**batch)
            raw_loss = outputs.loss
            loss = raw_loss / GRAD_ACCUM

        scaler.scale(loss).backward()
        running_raw_loss += raw_loss.item()
        accum_count += 1

        do_update = (step % GRAD_ACCUM == 0) or (step == len(train_loader))
        if do_update:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_step += 1

            progress_bar.set_postfix({"loss": f"{running_raw_loss / accum_count:.4f}"})
            running_raw_loss = 0.0
            accum_count = 0

    # validation loss도 answer-only labels 기준으로 계산됩니다.
    model.eval()
    val_loss = 0.0
    val_steps = 0
    with torch.no_grad():
        for vb in tqdm(valid_loader, desc=f"Epoch {epoch+1} [valid-loss]", unit="batch"):
            vb = {k: v.to(MODEL_DEVICE, non_blocking=True) for k, v in vb.items()}
            with torch.autocast(device_type="cuda", dtype=COMPUTE_DTYPE):
                val_loss += model(**vb).loss.item()
            val_steps += 1

    print(f"[Epoch {epoch+1}] answer-only valid loss: {val_loss / max(val_steps, 1):.4f}")

SAVE_DIR = os.path.join(OUTPUT_DIR, "qwen2_5_vl_3b_lora_v4")
model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
print("Saved:", SAVE_DIR)


Epoch 1 [train]:   0%|          | 0/6043 [00:00<?, ?batch/s]c:\SSAFY\AIChallenge\baseline\Lib\site-packages\torch\utils\checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
Epoch 1 [valid-loss]: 100%|██████████| 671/671 [08:13<00:00,  1.36batch/s]


[Epoch 1] answer-only valid loss: 0.0629
Saved: output\qwen2_5_vl_3b_lora_v4


# inference

30분~1시간 소요

#### 실습 참고 내용

    챕터4-1 RAG 기반 Customer Service AI 에이전트 개발
    - 데이터 파서 : langchain_core.output_parsers(), StrOutputParser()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [12]:
CHOICES = ["a", "b", "c", "d"]
choice_token_ids = []
for c in CHOICES:
    ids = processor.tokenizer.encode(c, add_special_tokens=False)
    if len(ids) != 1:
        raise RuntimeError(f"'{c}'가 단일 토큰이 아닙니다: {ids}. 이 경우 sequence scoring으로 바꾸세요.")
    choice_token_ids.append(ids[0])
choice_token_ids = torch.tensor(choice_token_ids, device=MODEL_DEVICE)
print("choice token ids:", choice_token_ids.tolist())


def predict_df(df, batch_size=1):
    """
    generate() 대신 assistant의 첫 토큰 logits에서 a/b/c/d만 직접 비교합니다.
    객관식 출력 파싱 실패를 없애고 추론도 더 빠르게 합니다.
    """
    model.eval()
    model.config.use_cache = True
    preds, all_logp = [], []

    for start in tqdm(range(0, len(df), batch_size), desc="Inference", unit="batch"):
        part = df.iloc[start:start+batch_size]
        texts, images = [], []

        for _, row in part.iterrows():
            with Image.open(os.path.join(DATA_DIR, row["path"])) as im:
                img = im.convert("RGB")

            user_text = build_mc_prompt(row["question"], row["a"], row["b"], row["c"], row["d"])
            messages = [
                {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
                {"role": "user", "content": [
                    {"type": "image", "image": img},
                    {"type": "text", "text": user_text},
                ]},
            ]

            text = processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
            texts.append(text)
            images.append(img)

        inputs = processor(
            text=texts,
            images=images,
            padding=True,
            return_tensors="pt",
        )
        inputs = {k: v.to(MODEL_DEVICE, non_blocking=True) for k, v in inputs.items()}

        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=COMPUTE_DTYPE):
            logits = model(**inputs).logits

        # 오른쪽 padding이므로 각 샘플의 마지막 유효 토큰 위치에서 다음 토큰을 평가
        last_idx = inputs["attention_mask"].sum(dim=1) - 1
        batch_idx = torch.arange(len(part), device=MODEL_DEVICE)
        next_token_logits = logits[batch_idx, last_idx]
        choice_scores = next_token_logits.index_select(dim=1, index=choice_token_ids)
        pred_idx = choice_scores.argmax(dim=1).tolist()
        preds.extend([CHOICES[i] for i in pred_idx])
        all_logp.extend(torch.log_softmax(choice_scores.float(), dim=1).cpu().tolist())

    return preds, all_logp


# 먼저 validation Accuracy를 확인합니다. 대회 metric과 동일한 지표가 중요합니다.
valid_preds, valid_logp = predict_df(valid_subset, batch_size=1)
valid_acc = np.mean(np.array(valid_preds) == valid_subset["answer"].astype(str).str.lower().to_numpy())
print(f"Validation accuracy: {valid_acc:.4f}")
valid_scores = valid_subset[["id", "path", "question", "answer"]].copy().reset_index(drop=True)
valid_scores["pred"] = valid_preds
for j, c in enumerate(CHOICES):
    valid_scores[f"logp_{c}"] = [row[j] for row in valid_logp]
valid_scores.to_csv(os.path.join(OUTPUT_DIR, "v4_valid_scores.csv"), index=False)

# test inference
preds, test_logp = predict_df(test_df, batch_size=1)
print("prediction distribution:", pd.Series(preds).value_counts(normalize=True).sort_index().to_dict())
test_scores = test_df[["id", "path", "question"]].copy().reset_index(drop=True)
test_scores["pred"] = preds
for j, c in enumerate(CHOICES):
    test_scores[f"logp_{c}"] = [row[j] for row in test_logp]
test_scores.to_csv(os.path.join(OUTPUT_DIR, "v4_test_scores.csv"), index=False)

sample_path = os.path.join(DATA_DIR, "sample_submission.csv")
if os.path.exists(sample_path):
    submission = pd.read_csv(sample_path)
    submission["answer"] = preds
else:
    submission = pd.DataFrame({"id": test_df["id"], "answer": preds})

SUBMISSION_PATH = os.path.join(OUTPUT_DIR, "submission_v4.csv")
submission.to_csv(SUBMISSION_PATH, index=False)
print("Saved", SUBMISSION_PATH)


choice token ids: [64, 65, 66, 67]


Inference: 100%|██████████| 671/671 [08:13<00:00,  1.36batch/s]


Validation accuracy: 0.9344


Inference: 100%|██████████| 6714/6714 [1:51:11<00:00,  1.01batch/s]

prediction distribution: {'a': 0.2563300565981531, 'b': 0.24247840333631218, 'c': 0.2487339886803694, 'd': 0.25245755138516535}
Saved output\submission_v4.csv


In [13]:
submission.head()


,id,answer
0,test_0001.jpg,b
1,test_0002.jpg,c
2,test_0003.jpg,d
3,test_0004.jpg,d
4,test_0005.jpg,d
